<a href="https://colab.research.google.com/github/joelrvas/pucp-ia-chatbot-work-01/blob/feature%2Fmelqui/Tarea_NLU_with_Rasa_no_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RASA

In [1]:
!python3 --version

Python 3.11.11


## DATA TO YML


In [2]:
! pip install ruamel.yaml

In [3]:
from ruamel.yaml import YAML
from ruamel.yaml.scalarstring import LiteralScalarString
import pandas as pd

def convert_to_yml(file_input, file_output):
  df = pd.read_csv(file_input).dropna(subset=["category", "text"])

  df["category"] = df["category"].astype(str)
  df["text"] = df["text"].astype(str).str.strip()

  nlu_data = {"version": "3.1", "nlu": []}

  for intent, sub_df in df.groupby("category"):
      # Agregar un salto de línea final para evitar que YAML use '|-'
      examples_text = "\n".join(f"- {item.strip()}" for item in sub_df["text"]) + "\n"

      intent_data = {
          "intent": intent,
          "examples": LiteralScalarString(examples_text)  # 🔹 Mantiene el formato correcto de bloque YAML
      }
      nlu_data["nlu"].append(intent_data)

  yaml = YAML()
  yaml.default_flow_style = False
  yaml.indent(mapping=2, sequence=4, offset=2)  # Ajustar indentación correcta

  with open(file_output, "w", encoding="utf-8") as f:
      yaml.dump(nlu_data, f)

  print(f"✅ Archivo YAML generado correctamente: {file_output}")
  return file_output



def convert_json_to_yml(file_input, file_output):
  import json
  from ruamel.yaml import YAML

  json_file = file_input
  with open(json_file, "r", encoding="utf-8") as f:
      data = json.load(f)

  intent_names = [intent for intent in data]

  # Crear la estructura YAML
  yaml_data = {
      "version": "3.1",
      "intents": intent_names
  }

  yaml = YAML()
  yaml.default_flow_style = False

  yaml_file = file_output
  with open(yaml_file, "w", encoding="utf-8") as f:
      yaml.dump(yaml_data, f)

  print(f"Archivo YAML generado: {yaml_file}")
  return yaml_file



In [4]:
!mkdir data

mkdir: cannot create directory ‘data’: File exists


In [5]:
! wget https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/refs/heads/master/banking_data/categories.json -O data/categories.json
! wget https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/refs/heads/master/banking_data/test.csv -O data/test.csv
! wget https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/refs/heads/master/banking_data/train.csv -O data/train.csv

--2025-03-18 01:38:20--  https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/refs/heads/master/banking_data/categories.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2036 (2.0K) [text/plain]
Saving to: ‘data/categories.json’

data/categories.jso 100%[===================>]   1.99K  --.-KB/s    in 0s      

2025-03-18 01:38:21 (32.5 MB/s) - ‘data/categories.json’ saved [2036/2036]

--2025-03-18 01:38:21--  https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/refs/heads/master/banking_data/test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP 

In [6]:
nlu_train = convert_to_yml("data/train.csv", "data/training_data.yml")

✅ Archivo YAML generado correctamente: data/training_data.yml


In [7]:
nlu_test = convert_to_yml("data/test.csv", "data/test_data.yml")

✅ Archivo YAML generado correctamente: data/test_data.yml


In [8]:
nlu_intent = convert_json_to_yml("data/categories.json", "data/domain.yml")

Archivo YAML generado: data/domain.yml


In [9]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [10]:
!conda create --name myenv python=3.10

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 25.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/myenv

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _libgcc_mutex      conda-forge/linux-64::_libgcc_mutex-0.1-conda_forge 
  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-2_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-h4bc722e_7 
  ca-certificates    conda-forge/linux-64::ca-certificates-2025.1.31-hbcca054_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.43-h712a8e2_4 
  libffi             conda-forge/linux-64::libffi-3.4.6-h2dba641_0 
  libgcc             conda-forge/linux-64::libgcc-14.2.0-h767d61c_2 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-14.2.0-h69a702

## Requerimientos

In [11]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
python3 -m pip install rasa==3.6.21 rasa[spacy]==3.6.21

  Using cached rasa-3.6.21-py3-none-any.whl.metadata (28 kB)
  Using cached CacheControl-0.12.14-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached PyJWT-2.10.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached SQLAlchemy-1.4.54-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached absl_py-1.4.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached aio_pika-8.2.3-py3-none-any.whl.metadata (9.5 kB)
  Using cached aiogram-2.25.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached aiohttp-3.9.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.5 kB)
  Using cached APScheduler-3.9.1.post1-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached attrs-22.1.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached boto3-1.37.14-py3-none-any.whl.metadata (6.7 kB)
  Using cached certifi-2025.1.31-py3-none-any.whl.metadata (2.5 kB)
  Using cached colorclass-2.2.2-py2.py3-none-any.whl.metadata (5.2 kB)
  Usin

In [12]:
%mkdir -p model_trained
%mkdir -p model_tested

In [13]:
%ls

condacolab_install.log  model_tested/   nlu-20250314-202523-flat-convertible.tar.gz  sample_data/
data/                   model_trained/  nlu_config.yml


## Corpus

**Adapted from:** https://github.com/cedextech/rasa-chatbot-templates/tree/master/01_smalltalk_bot/src

In [14]:
#@title nlu_config.yml
%%writefile nlu_config.yml
recipe: default.v1

language: en

pipeline:
- name: WhitespaceTokenizer
- name: CountVectorsFeaturizer
- name: DIETClassifier
  epochs: 100
  constrain_similarities: true
  model_confidence: softmax

Overwriting nlu_config.yml


## Entrenamiento del modelo

In [15]:
%rm -rf .rasa/cache

In [16]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
rasa telemetry disable

/usr/local/envs/myenv/lib/python3.10/site-packages/rasa/core/tracker_store.py:1044: MovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base: DeclarativeMeta = declarative_base()
/usr/local/envs/myenv/lib/python3.10/site-packages/rasa/shared/utils/validation.py:134: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
/usr/local/envs/myenv/lib/python3.10/site-packages/pkg_resources/__init__.py:3117: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implement

In [17]:
#%%shell
#eval "$(conda shell.bash hook)"
#conda activate myenv
#rasa train nlu --nlu data/training_data.yml --config nlu_config.yml --domain data/domain.yml --out model_trained

In [18]:
%ls model_trained

nlu-20250314-202523-flat-convertible.tar.gz


## Prueba del modelo

In [19]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
rasa test nlu --model model_trained/nlu-20250314-202523-flat-convertible.tar.gz --nlu data/test_data.yml \
               --config nlu_config.yml --domain data/domain.yml --out model_tested --no-plot

/usr/local/envs/myenv/lib/python3.10/site-packages/rasa/core/tracker_store.py:1044: MovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base: DeclarativeMeta = declarative_base()
/usr/local/envs/myenv/lib/python3.10/site-packages/rasa/shared/utils/validation.py:134: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
/usr/local/envs/myenv/lib/python3.10/site-packages/pkg_resources/__init__.py:3117: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implement

In [20]:
%ls -l model_tested

total 84
-rw-r--r-- 1 root root 63427 Mar 18 01:41 intent_errors.json
-rw-r--r-- 1 root root 17562 Mar 18 01:41 intent_report.json


In [21]:
!head model_tested/intent_errors.json --lines=20

[
  {
    "text": "I requested a refund, and never received it. What can I do?",
    "intent": "Refund_not_showing_up",
    "intent_prediction": {
      "name": "request_refund",
      "confidence": 0.5915908813476562
    }
  },
  {
    "text": "I was supposed to get a purchase refunded but I don't see the money in my account",
    "intent": "Refund_not_showing_up",
    "intent_prediction": {
      "name": "request_refund",
      "confidence": 0.6145465970039368
    }
  },
  {
    "text": "I bought something and returned it and the money from the return isn't in my account.",
    "intent": "Refund_not_showing_up",


In [22]:
!tail model_tested/intent_errors.json --lines=20

      "confidence": 0.9279323816299438
    }
  },
  {
    "text": "Is there a fee for exchanging cash?",
    "intent": "wrong_exchange_rate_for_cash_withdrawal",
    "intent_prediction": {
      "name": "exchange_charge",
      "confidence": 0.6339101791381836
    }
  },
  {
    "text": "I feel like too much money was taken during my currency exchange.",
    "intent": "wrong_exchange_rate_for_cash_withdrawal",
    "intent_prediction": {
      "name": "card_payment_wrong_exchange_rate",
      "confidence": 0.7637776136398315
    }
  }
]

In [23]:
!head model_tested/intent_report.json --lines=20

{
  "declined_cash_withdrawal": {
    "precision": 0.8163265306122449,
    "recall": 1.0,
    "f1-score": 0.898876404494382,
    "support": 40,
    "confused_with": {}
  },
  "why_verify_identity": {
    "precision": 0.875,
    "recall": 0.7,
    "f1-score": 0.7777777777777777,
    "support": 40,
    "confused_with": {
      "verify_my_identity": 7,
      "unable_to_verify_identity": 5
    }
  },
  "top_up_limits": {
    "precision": 0.975,


In [24]:
!tail model_tested/intent_report.json --lines=30

  "beneficiary_not_allowed": {
    "precision": 0.8837209302325582,
    "recall": 0.95,
    "f1-score": 0.9156626506024096,
    "support": 40,
    "confused_with": {
      "failed_transfer": 1,
      "transfer_into_account": 1
    }
  },
  "accuracy": 0.9090613835660929,
  "macro avg": {
    "precision": 0.9112592927141843,
    "recall": 0.909090909090909,
    "f1-score": 0.9087318001094878,
    "support": 3079
  },
  "weighted avg": {
    "precision": 0.9112463144032129,
    "recall": 0.9090613835660929,
    "f1-score": 0.9087102774723035,
    "support": 3079
  },
  "micro avg": {
    "precision": 0.9090613835660929,
    "recall": 0.9090613835660929,
    "f1-score": 0.9090613835660929,
    "support": 3079
  }
}

## Probando el modelo mensaje por mensaje

In [25]:
%mkdir -p model_trained

In [26]:
%ls -l model_trained

total 24336
-rw-r--r-- 1 root root 24919634 Mar 18 01:17 nlu-20250314-202523-flat-convertible.tar.gz


In [ ]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
rasa shell -m model_trained/nlu-20250314-202523-flat-convertible.tar.gz